In [2]:
import pandas as pd
roll_number = "1024160078"
last_two_digits = [int(roll_number[-2]), int(roll_number[-1])]

categories = ["billing", "account", "general"]

fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": ["fee", "cost", "price", "charge"],
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": ["password", "reset", "login"],
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": ["hours", "timing", "open", "general"],
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": ["pay", "payment", "fee", "upi", "card", "net banking"],
        "category": "billing"
    }
]

# Construct entries from last two digits
new_entries = []

for digit in last_two_digits:
    category = categories[digit % 3]

    if digit == 7:
        entry = {
            "question": "how do I update my registered mobile number",
            "answer": "Go to Account Settings and update your registered mobile number.",
            "keywords": ["mobile", "number", "update", "registered"],
            "category": category
        }
    else:
        entry = {
            "question": "where can I find general account information",
            "answer": "You can find general information in the Help and Information section.",
            "keywords": ["general", "information", "help", "details"],
            "category": category
        }

    new_entries.append(entry)

all_entries = fixed_entries + new_entries

df = pd.DataFrame(all_entries)

print("Q1 - Final DataFrame:")
print(df)

Q1 - Final DataFrame:
                                       question  \
0                        what is the annual fee   
1                         how to reset password   
2                   what are your working hours   
3                         how can i pay the fee   
4   how do I update my registered mobile number   
5  where can I find general account information   

                                              answer  \
0                          The annual fee is Rs 500.   
1                   Go to Settings > Reset Password.   
2                          We are open 9 AM to 5 PM.   
3         You can pay via UPI, card, or net banking.   
4  Go to Account Settings and update your registe...   
5  You can find general information in the Help a...   

                                      keywords category  
0                   [fee, cost, price, charge]  billing  
1                     [password, reset, login]  account  
2               [hours, timing, open, general]  gener

In [3]:
def score_query(query, df):
    query_words = set(query.lower().split())
    results = []

    for _, row in df.iterrows():
        matched_keywords = [
            word for word in row["keywords"]
            if word.lower() in query_words
        ]

        score = len(matched_keywords)

        if score > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "score": score,
                "matched_keywords": matched_keywords
            })

    results.sort(key=lambda x: x["score"], reverse=True)

    return results


query = input("\nQ2 - Enter your query: ")
results = score_query(query, df)

print("\nMatching entries ranked by confidence:")

if results:
    for result in results:
        print(result)
else:
    print("No matching entries found.")


Q2 - Enter your query: fee payment

Matching entries ranked by confidence:
{'question': 'how can i pay the fee', 'answer': 'You can pay via UPI, card, or net banking.', 'category': 'billing', 'score': 2, 'matched_keywords': ['payment', 'fee']}
{'question': 'what is the annual fee', 'answer': 'The annual fee is Rs 500.', 'category': 'billing', 'score': 1, 'matched_keywords': ['fee']}


In [4]:
def same_category(category_name, df):
    return df[df["category"].str.lower() == category_name.lower()]

category_name = input("\nQ3 - Enter category: ")

print("\nEntries belonging to category:")
print(same_category(category_name, df))


Q3 - Enter category: account

Entries belonging to category:
                                      question  \
1                        how to reset password   
4  how do I update my registered mobile number   

                                              answer  \
1                   Go to Settings > Reset Password.   
4  Go to Account Settings and update your registe...   

                               keywords category  
1              [password, reset, login]  account  
4  [mobile, number, update, registered]  account  


In [5]:
print("\nQ4 - Available questions:")
for i, question in enumerate(df["question"], start=0):
    print(i, question)

index = int(input("Enter the index of the entry: "))
new_keyword = input("Enter a new keyword: ")

df.at[index, "keywords"].append(new_keyword)

filename = f"{roll_number}_faq_data.csv"

# Convert keyword lists to strings before saving
csv_df = df.copy()
csv_df["keywords"] = csv_df["keywords"].apply(lambda x: ", ".join(x))

csv_df.to_csv(filename, index=False)

print(f"Updated DataFrame saved to {filename}")


Q4 - Available questions:
0 what is the annual fee
1 how to reset password
2 what are your working hours
3 how can i pay the fee
4 how do I update my registered mobile number
5 where can I find general account information
Enter the index of the entry: 0
Enter a new keyword: subscription
Updated DataFrame saved to 1024160078_faq_data.csv


In [6]:
print("\nQ5 - Number of FAQ entries per category:")
print(df.groupby("category").size())


Q5 - Number of FAQ entries per category:
category
account    2
billing    2
general    2
dtype: int64


In [12]:
def score_query_with_ties(query, df):
    query_words = set(query.lower().split())
    results = []

    for _, row in df.iterrows():
        matched_keywords = [
            word for word in row["keywords"]
            if word.lower() in query_words
        ]

        score = len(matched_keywords)

        if score > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "score": score,
                "matched_keywords": matched_keywords
            })

    if not results:
        return []

    highest_score = max(result["score"] for result in results)

    tied_results = [
        result for result in results
        if result["score"] == highest_score
    ]

    return tied_results

In [11]:
print("\nQ6 - Query producing a tie:")
tie_query = "fee"

tie_results = score_query_with_ties(tie_query, df)

for result in tie_results:
    print(result)

print("\nQ6 - Query without a tie:")
no_tie_query = "password"

no_tie_results = score_query_with_ties(no_tie_query, df)

for result in no_tie_results:
    print(result)


Q6 - Query producing a tie:
{'question': 'what is the annual fee', 'answer': 'The annual fee is Rs 500.', 'category': 'billing', 'score': 1, 'matched_keywords': ['fee']}
{'question': 'how can i pay the fee', 'answer': 'You can pay via UPI, card, or net banking.', 'category': 'billing', 'score': 1, 'matched_keywords': ['fee']}

Q6 - Query without a tie:
{'question': 'how to reset password', 'answer': 'Go to Settings > Reset Password.', 'category': 'account', 'score': 1, 'matched_keywords': ['password']}
